# Cruzamento de Dados - Check-in Aulas Semanais
### Objetivo: Cruzar os dados de email dos Check-ins das Aulas Semanais para buscar o último telefone válido dos leads.

In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

#PRODUCT Ids
CRONOGRAMA_ID = '3310674'

#Subdomains do Hotmart Club
CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'

#Aulas de Conversação 
AULAS_DE_CONVERSACAO_SUBDOMAIN = 'aulasdeconversacao-vpqdjd'

#Ingles Express Travel
INGLES_EXPRESS_TRAVEL_SUBDOMAIN = 'inglesexpresstravel'

#Intensivão do Inglês
INTENSIVAO_DO_INGLES_BASICO_SUBDOMAIN = 'intensivaodoinglesbasico'

#Aulas de Conversação Particulares
AULAS_DE_CONVERSACAO_PARTICULARES_SUBDOMAIN = 'aulasdeconversacao-avjbpp'

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df, 
    timestamp_ms_to_datetime
)

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários tmb
from tmb_utils import TMB   

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

print('✓ Importações concluídas com sucesso!')

✓ Importações concluídas com sucesso!


# Buscando Check-ins na Aula Definida


In [2]:
CLASS_DATE = '2025-12-09 22:07:00+00'

df_checkins = db.execute_query_from_file('select_as_checkins.sql', {'class_date': CLASS_DATE})
#df_checkins = db.execute_query_from_file('select_as_all_checkins.sql')

df_checkins['email'] = clean_and_lower_column(df_checkins, 'email')
df_checkins.shape

(4775, 13)

In [9]:
df_as = db.execute_query_from_file('select_complete_lead_data_from_campaign', params={'conversion_type_id': '1', 'campaign_id': 'isca-aulassemanais'})

"""
Cell generated by Data Wrangler.
"""
from datetime import datetime

def clean_data(df_as):
    df_as['last_conversion_date'] = pd.to_datetime(df_as['last_conversion_date'], format='%Y-%m-%d')
    # Filter rows based on column: 'last_conversion_date'
    df_as = df_as[df_as['last_conversion_date'] < datetime.strptime('2025-12-09T22:30:05.601Z', '%Y-%m-%dT%H:%M:%S.%fZ')]
    # Drop rows with missing data in column: 'phone'
    df_as = df_as.dropna(subset=['phone'])
    # Drop duplicate rows in column: 'lead_id'
    df_as = df_as.drop_duplicates(subset=['lead_id'])
    # Drop duplicate rows in column: 'phone'
    df_as = df_as.drop_duplicates(subset=['phone'])

    df_as['phone'] = process_phone_string(df_as, 'phone')
    # Drop columns: 'utm_source', 'origem' and 8 other columns
    df_as = df_as.drop(columns=['utm_source', 'origem', 'utm_campaign', 'utm_content', 'utm_medium', 'utm_term', 'faixa_de_idade', 'renda', 'ja_fez_outro_curso', 'last_campaign_id'])
    return df_as

df_as = clean_data(df_as.copy())
df_as.head()


,lead_id,last_conversion_date,name,email,phone,whatsapp_format,renda_qualificada,qualificado_geral
0,000d9334-4375-4bd2-987d-d31dd697be45,2025-05-05,Maria do Amparo da Silva Medeiros,mariadoamparo327@gmail.com,5584988459837,+558488459837,False,False
1,0010c589-a7f6-40e9-ad92-f0baa2895977,2025-04-16,Adriana,abssaguiga@yahoo.com.br,5521998518990,+5521998518990,False,False
2,0012a2c8-f5f9-4eb5-99ec-1fb4e3fe4281,2025-08-11,Douglas,douglasmustafa10@gmail.com,351935765282,351935765282,False,False
3,0029d21c-a1f2-47b2-b9ec-d62825386c7b,2025-05-22,Paloma Godoi,paloma.unifoco@gmail.com,5512988581400,+5512988581400,False,False
4,0036d725-931a-40f3-afa7-00724073bba5,2025-03-22,Debora Fabiani,deborafabiani83@gmail.com,5524981187137,+5524981187137,True,True


## Buscando Emails e Phones para Cruzamento

In [4]:
df_emails = db.execute_query_from_file('select_all_emails.sql')

df_emails['email'] = clean_and_lower_column(df_emails, 'email')

print('df_emails.shape', df_emails.shape)

df_phones = db.execute_query_from_file('select_leads_last_phone.sql')

print('df_phones.shape', df_phones.shape)

df_names = db.execute_query_from_file('select_all_names.sql')

print('df_names.shape', df_names.shape)

df_emails.shape (326895, 6)
df_phones.shape (402605, 11)
df_names.shape (384687, 2)


### Cruzando E-mails com df_checkin para validar lead_id

In [5]:
df_checkins_emails = pd.merge(df_checkins, df_emails, on='email', how='left')

### Cruzando checkins com Phones

In [6]:
df_checkins_phones = pd.merge(df_checkins_emails, df_phones, on='lead_id', how='left')

# Cruzando Checkins com Names

In [7]:
df_checkins_names = pd.merge(df_checkins_phones, df_names, on='lead_id', how='left')

## Limpeza de Dados


In [8]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df_checkins_phones):
    # Drop columns: 'session_id', 'user_id' and 15 other columns
    df_checkins_phones = df_checkins_phones.drop(columns=['session_id', 'user_id', 'created_at_x', 'class_date', 'is_active', 'expiration_date', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_term', 'utm_content', 'session_type', 'id_x', 'active_id', 'created_at_y', 'checked', 'id_y', 'raw_phone_input', 'phone_type', 'ddd', 'ddi', 'is_valid', 'created_at', 'reprocessed'])
    # Rename column 'created_at_x' to 'check_in_date'
    df_checkins_phones = df_checkins_phones.rename(columns={'created_at_x': 'check_in_date'})
    # Drop rows with missing data in column: 'formatted_phone'
    df_checkins_phones = df_checkins_phones.dropna(subset=['formatted_phone'])
    # Drop duplicate rows in column: 'lead_id'
    df_checkins_phones = df_checkins_phones.drop_duplicates(subset=['lead_id'])
    # Drop duplicate rows in column: 'formatted_phone'
    df_checkins_phones = df_checkins_phones.drop_duplicates(subset=['formatted_phone'])

    df_checkins_phones['phone'] = process_phone_string(df_checkins_phones, 'formatted_phone')
    return df_checkins_phones

df_checkins_names_clean = clean_data(df_checkins_names.copy())
df_checkins_names_clean.head()

,email,lead_id,formatted_phone,whatsapp_format,name,phone
0,izabeeeelsoares@gmail.com,63c054b0-5b9c-4f9f-aa6b-9c336ce40c2f,+5518996904673,+5518996904673,Izabel,5518996904673
2,fatimaamecina5@gmail.com,6e75a625-ac62-491c-8ab6-40cf88faa82c,+5511952476403,+5511952476403,None,5511952476403
3,cantuariaalvesap@gmail.com,403401ef-a22d-4a13-bb05-c16e420ceca1,+5562998319136,+556298319136,Ana Alves,5562998319136
4,mercifardin2@gmail.com,1aea36e5-27b0-479a-85c2-bf4c8b58370f,+5527998662926,+5527998662926,Merci Fardin,5527998662926
5,antoninia10@yahoo.com.br,4a8f0b08-462f-4e61-9ada-7cf7d3b5621d,+556492939443,+556492939443,Antonínia Borges dos Anjos Cunha,556492939443


In [11]:
df_checkins_names_clean = pd.concat([df_checkins_names_clean, df_as])

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_checkins_names_clean):
    # Drop rows with missing data in column: 'phone'
    df_checkins_names_clean = df_checkins_names_clean.dropna(subset=['phone'])
    # Drop duplicate rows in column: 'lead_id'
    df_checkins_names_clean = df_checkins_names_clean.drop_duplicates(subset=['lead_id'])
    # Drop duplicate rows in column: 'email'
    df_checkins_names_clean = df_checkins_names_clean.drop_duplicates(subset=['email'])
    # Drop duplicate rows in column: 'phone'
    df_checkins_names_clean = df_checkins_names_clean.drop_duplicates(subset=['phone'])
    return df_checkins_names_clean

df_checkins_names_clean = clean_data(df_checkins_names_clean.copy())
df_checkins_names_clean.head()

,email,lead_id,formatted_phone,whatsapp_format,name,phone,last_conversion_date,renda_qualificada,qualificado_geral
0,izabeeeelsoares@gmail.com,63c054b0-5b9c-4f9f-aa6b-9c336ce40c2f,+5518996904673,+5518996904673,Izabel,5518996904673,NaT,NaN,NaN
2,fatimaamecina5@gmail.com,6e75a625-ac62-491c-8ab6-40cf88faa82c,+5511952476403,+5511952476403,None,5511952476403,NaT,NaN,NaN
3,cantuariaalvesap@gmail.com,403401ef-a22d-4a13-bb05-c16e420ceca1,+5562998319136,+556298319136,Ana Alves,5562998319136,NaT,NaN,NaN
4,mercifardin2@gmail.com,1aea36e5-27b0-479a-85c2-bf4c8b58370f,+5527998662926,+5527998662926,Merci Fardin,5527998662926,NaT,NaN,NaN
5,antoninia10@yahoo.com.br,4a8f0b08-462f-4e61-9ada-7cf7d3b5621d,+556492939443,+556492939443,Antonínia Borges dos Anjos Cunha,556492939443,NaT,NaN,NaN


## Buyers

In [12]:
df_buyers = db.execute_query_from_file('select_generic_sales', params={'product_name': 'Cronograma dos Fluentes'})

df_buyers['phone'] = process_phone_string(df_buyers, 'phone')

print('Df Shape buyers:', df_buyers.shape)

Df Shape buyers: (38838, 9)


In [13]:
df_hotmart = load_raw_data('hotmart.csv', sep=';')

def clean_data(hotmart_transactions):
    # Drop columns: 'Nome do Produtor', 'Documento do Produtor' and 49 other columns
    hotmart_transactions = hotmart_transactions.drop(columns=['Nome do Produtor', 'Documento do Produtor', 'Nome do Afiliado', 'Transação', 'Meio de Pagamento', 'Origem', 'Moeda', 'Preço do Produto', 'Moeda.1', 'Preço da Oferta', 'Taxa de Câmbio', 'Moeda.2', 'Preço Original', 'Número da Parcela', 'Data de Venda', 'Data de Confirmação', 'Status', 'Documento', 'CEP', 'Cidade', 'Estado', 'Bairro', 'País', 'Endereço', 'Número', 'Complemento', 'chave', 'Código do Produto', 'Código da Afiliação', 'Tipo de Pagamento', 'Período Grátis', 'Tem co-produção', 'Venda feita como', 'Preço Total', 'Tipo pagamento oferta', 'Taxa de Câmbio Real', 'Preço Total Convertido', 'Quantidade de itens', 'Oferta de Upgrade', 'Cupom', 'Moeda.3', 'Valor que você recebeu convertido', 'Data Vencimento', 'Taxa de Câmbio do valor recebido', 'Instagram', 'Origem da venda', 'Moeda de recebimento', 'Faturamento líquido', 'Código do assinante', 'Nota Fiscal', 'Valor do frete bruto'])
    # Filter rows based on column: 'Recorrência'
    hotmart_transactions = hotmart_transactions[(hotmart_transactions['Recorrência'] == 1) | (hotmart_transactions['Recorrência'].isna())]
    hotmart_transactions['DDD'] = process_phone_string(hotmart_transactions, 'DDD')
    hotmart_transactions['Telefone'] = process_phone_string(hotmart_transactions, 'Telefone')
    hotmart_transactions['phone'] = hotmart_transactions.apply(lambda x: x['Telefone'] if pd.isna(x['DDD']) else f'55{x["DDD"]}{x["Telefone"]}', axis=1)
    # Drop column: 'DDD'
    hotmart_transactions = hotmart_transactions.drop(columns=['DDD', 'Telefone'])

    # Drop column: 'Recorrência'
    hotmart_transactions = hotmart_transactions.drop(columns=['Recorrência'])
    # Rename column 'Nome do Produto' to 'product_name'
    hotmart_transactions = hotmart_transactions.rename(columns={'Nome do Produto': 'product_name', "Nome": "name", "Email": "email","Código de Oferta": "offer_id" })
    # Drop column: 'Origem de Checkout'
    hotmart_transactions = hotmart_transactions.drop(columns=['Origem de Checkout'])
    hotmart_transactions = hotmart_transactions[hotmart_transactions['product_name'] == 'Cronograma dos Fluentes ']
    return hotmart_transactions

df_hotmart = clean_data(df_hotmart.copy())


df_tmb = load_raw_data('tmb.csv', sep=';')

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Drop columns: 'Produtor', 'Pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['Produtor', 'Pedido', 'Endereço completo', 'Logradouro', 'Número', 'Bairro', 'Complemento', 'CEP', 'Estado', 'Cidade', 'País', 'Status', 'Status Financeiro', 'Status Cancelamento', 'Ticket (R$)', 'Modalidade de Contrato', 'Criado Em', 'Data Efetivado', 'Data Cancelado', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_content', 'utm_last_source', 'utm_last_medium', 'utm_last_campaign', 'utm_last_content', 'Renovação automatica', 'Data de Renovação Prevista', 'Renovação enviada', 'Oferta'])
    # Drop column: 'Cliente CPF'
    df_tmb = df_tmb.drop(columns=['Cliente CPF'])
    # Rename column 'Produto' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'Produto': 'offer_id', "Cliente Nome": "name", "Cliente Email": "email", "Telefone": "phone"})

    df_tmb['phone'] = process_phone_string(df_tmb, 'phone')
    
    return df_tmb

df_tmb = clean_data(df_tmb.copy())

In [15]:
df_hotmart_pt_1 =  load_raw_data('hotmart_pt_1.csv', data_dir='../data/interim')

df_hotmart_pt_2 = load_raw_data('hotmart_pt_2.csv', data_dir='../data/interim')

df_hotmart_pt_3 = flatten_list_to_df(hotmart.get_sales_history(
    start_date='2025-12-01',
    end_date='2025-12-31',
))

df_hotmart_2 = pd.concat([df_hotmart_pt_1, df_hotmart_pt_2, df_hotmart_pt_3])

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_hotmart):
    # Filter rows based on column: 'purchase_recurrency_number'
    df_hotmart = df_hotmart[(df_hotmart['purchase_recurrency_number'] == 1) | (df_hotmart['purchase_recurrency_number'].isna())]
    # Drop columns: 'purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total' and 22 other columns
    df_hotmart = df_hotmart.drop(columns=['purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total', 'purchase_hotmart_fee_fixed', 'purchase_hotmart_fee_percentage', 'purchase_hotmart_fee_base', 'purchase_is_subscription', 'purchase_offer_payment_mode', 'purchase_tracking_source', 'purchase_payment_type', 'purchase_payment_installments_number', 'purchase_commission_as', 'purchase_warranty_expire_date', 'purchase_order_date', 'purchase_price_value', 'purchase_price_currency_code', 'producer_ucode', 'product_id', 'producer_name', 'buyer_ucode', 'purchase_tracking_source_sck', 'purchase_recurrency_number', 'purchase_tracking_external_code', 'purchase_payment_method', 'purchase_status'])
    # Rename column 'purchase_transaction' to 'transaction'
    df_hotmart = df_hotmart.rename(columns={'purchase_transaction': 'transaction'})
    # Rename column 'purchase_offer_code' to 'offer_code'
    df_hotmart = df_hotmart.rename(columns={'purchase_offer_code': 'offer_code'})
    # Rename column 'purchase_approved_date' to 'approved_at'
    df_hotmart = df_hotmart.rename(columns={'purchase_approved_date': 'approved_at'})
    # Rename column 'buyer_name' to 'name'
    df_hotmart = df_hotmart.rename(columns={'buyer_name': 'name'})
    # Rename column 'buyer_email' to 'email'
    df_hotmart = df_hotmart.rename(columns={'buyer_email': 'email'})
    # Rename column 'approved_at' to 'approval_date'
    df_hotmart = df_hotmart.rename(columns={'approved_at': 'approval_date'})
    df_hotmart['approval_date'] = timestamp_ms_to_datetime(df_hotmart, 'approval_date')
    return df_hotmart

df_hotmart_2 = clean_data(df_hotmart_2.copy())
df_hotmart_2.head()

df_tmb_2 = flatten_list_to_df(tmb.get_pedidos(page_size=1000))

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Filter rows based on column: 'status_pedido'
    df_tmb = df_tmb[df_tmb['status_pedido'] == "Efetivado"]
    # Drop columns: 'produtor', 'status_pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['produtor', 'status_pedido', 'documento', 'valor_total', 'parcelas', 'valor_entrada', 'valor_parcela', 'taxa_administracao', 'endereco_completo', 'criado_em', 'telefone', 'produto_id', 'nascimento', 'produtor_id', 'utm_source', 'utm_medium', 'utm_content', 'utm_campaign', 'utm_last_source', 'melhor_dia_pagamento', 'utm_last_medium', 'utm_last_content', 'utm_last_campaign', 'pais', 'cep', 'endereco_estado', 'endereco_cidade', 'endereco_bairro', 'endereco_logradouro', 'endereco_numero', 'endereco_complemento'])
    # Rename column 'lancamento' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'lancamento': 'offer_code'})
    # Rename column 'pedido_id' to 'transaction'
    df_tmb = df_tmb.rename(columns={'pedido_id': 'transaction'})
    # Rename column 'cliente' to 'name'
    df_tmb = df_tmb.rename(columns={'cliente': 'name'})
    # Rename column 'data_efetivado' to 'approval_date'
    df_tmb = df_tmb.rename(columns={'data_efetivado': 'approval_date'})
    # Change column type to datetime64[ns] for column: 'approval_date'
    df_tmb = df_tmb.astype({'approval_date': 'datetime64[ns]'})
    return df_tmb

df_tmb_2 = clean_data(df_tmb_2.copy())
df_tmb_2.head()

✓ Concluído: 18 página(s) processada(s). Total de 866 itens obtidos.)
✓ Concluído: 1 página(s) processada(s). Total de 100 pedidos obtidos.


,offer_code,transaction,name,email,approval_date,status_financeiro
0,COM-BF25-alunos-Oferta Negociação,1918767,Matheus Ruas,matheusruas@hotmail.com,2025-12-02 18:53:01.355239,Adimplente
1,COM-BF25-alunos-Oferta Negociação,1917023,KLEBER FERREIRA DO VALE,klebervaleadvogado@outlook.com,2025-12-02 19:00:26.471830,Adimplente
2,COM-BF25-alunos-Oferta Negociação,1915079,Kyara Costa de Azevedo,kyky338.157@gmail.com,2025-12-01 10:55:35.356717,Adimplente
3,COM-BF25-Oferta Downsell-3,1914968,Bruno Luis Reis Intaschi,brunointaschi@hotmail.com,2025-12-01 11:21:29.853887,Adimplente
4,COM-BF25-Oferta Downsell-3,1914804,Joana Darc Santos Almeida Oliveira,oliveiragiseli071@gmail.com,2025-12-01 09:30:20.814242,Adimplente


In [16]:
# Lead Ids Set
lead_ids_to_exclude = set()
lead_ids_to_exclude = lead_ids_to_exclude.union(set(df_buyers['lead_id'].unique()))

# Emails Set
emails_to_exclude = set()
emails_to_exclude = emails_to_exclude.union(set(df_buyers['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_tmb['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_hotmart['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_hotmart_2['email'].unique()))
emails_to_exclude = emails_to_exclude.union(set(df_tmb_2['email'].unique()))

# Phone Set
phones_to_exclude = set()
phones_to_exclude = phones_to_exclude.union(set(df_buyers['phone'].unique()))
phones_to_exclude = phones_to_exclude.union(set(df_tmb['phone'].unique()))
phones_to_exclude = phones_to_exclude.union(set(df_hotmart['phone'].unique()))

In [17]:
df_checkins_names_clean = df_checkins_names_clean[~df_checkins_names_clean['lead_id'].isin(lead_ids_to_exclude)]
df_checkins_names_clean = df_checkins_names_clean[~df_checkins_names_clean['email'].isin(emails_to_exclude)]
df_checkins_names_clean = df_checkins_names_clean[~df_checkins_names_clean['phone'].isin(phones_to_exclude)]
df_checkins_names_clean.head()
df_checkins_names_clean.shape


(94385, 9)

In [18]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df_checkins_names_clean):
    # Drop rows with missing data in column: 'last_conversion_date'
    df_checkins_names_clean = df_checkins_names_clean.dropna(subset=['last_conversion_date'])
    df_checkins_names_clean = df_checkins_names_clean.sort_values(
        ['last_conversion_date'],
        ascending=[False]
    ).head(5000)
    return df_checkins_names_clean

df_checkins_names_clean_1 = clean_data(df_checkins_names_clean.copy())
df_checkins_names_clean_1.head()

,email,lead_id,formatted_phone,whatsapp_format,name,phone,last_conversion_date,renda_qualificada,qualificado_geral
16529,dejaniranunes123@gmail.com,76d0635c-a461-44ba-9066-0f7aad3af3b8,NaN,+556291701416,Dejanira Nunes,5562991701416,2025-12-09,False,False
82213,jullyoliveira7@gmail.com,75cbb120-3843-4a4a-a09a-5a9d29a5bf68,NaN,+5513996036983,Juliana,5513996036983,2025-12-09,False,False
43982,josericardooliveiralima08@gmail.com,039ecdd5-449c-43a5-acfe-64ee24156253,NaN,+555531997085,José Ricardo de Oliveira Lima,555531997085,2025-12-09,False,False
43998,meririvera@hotmail.com,0879dcf6-9e90-4bfd-98f5-75eaad8a11db,NaN,+5513997262667,Meri meita,5513997262667,2025-12-09,False,False
44002,odairteixeira2022@gmail.com,0ac7d0fa-952d-4ea2-9874-857f49fbb2cf,NaN,+557998284751,Odair Teixeira,5579998284751,2025-12-09,False,False
